# Global centralized training featurs

In [ ]:
import numpy as np
import pandas as pd
import sqlalchemy as sqla
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score,  make_scorer,  make_scorer, average_precision_score, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import shap
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from skopt.callbacks import DeadlineStopper
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
import joblib
import matplotlib.pyplot as plt

In [ ]:

IBAN_COL   = 'Account'
TS_COL     = 'Timestamp'
LABEL_COL  = 'proxy_label' 
HOLDING_PSP_COL = 'To Bank'
DECLARING_PSP_COL = 'From Bank'
N          = 1


In [ ]:
out = pd.read_csv(r'./fraud_class_memoire/dataset/fake_fncrf.csv',on_bad_lines='warn', engine='python')
print(len(out['bank'].unique()))
out['bank'] = out['bank'].astype(str).str.strip()
out = out[out['bank'] != '70']


In [ ]:
summary = out.groupby(['bank', 'proxy_label']).size().unstack(fill_value=0)
summary['fp_ratio'] = summary.get('Faux Positif', 0) / summary.get('Fraudeur', 1)
summary.sort_values('fp_ratio', ascending=False).head(15)

In [ ]:

combined = out

In [ ]:
combined.columns

In [ ]:
combined['Is Laundering'].value_counts()

In [ ]:
combined = combined.drop(columns=['Is Laundering', 'nb.currency','delta.t', 'currency.mismatch', 'is.self.transfer', 'is.intra.bank',
       'log.amount', 'is.round.amount', 'hour.of.day', 'day.of.week',
       'is.off.hours', 'nb.distinct.to.bank_cum', 'nb.distinct.from.bank_cum',
       'nb.distinct.payfmt_cum', 'top.1.holder.RC', 'top.1.holder.SC',
       'nb.iban.holder', 'nb.events.holder', 'top.1.declaring.RC',
       'top.1.declaring.SC', 'nb.iban.declaring', 'nb.events.declaring',
       'fan.out', 'fan.in', 'fan.ratio', 'key', 'y_pred', 'bank'])

In [ ]:
combined.head()

##### We look at correlations 

In [ ]:
combined['proxy_label'].value_counts()

In [ ]:
combined['is_first_event'] = combined.groupby(IBAN_COL).cumcount() == 0

In [ ]:
def create_sliding(combined):
    # 1. Unify categorical dtype detection (object + category)
    cat_features = [c for c in combined.columns 
                    if c not in [IBAN_COL, TS_COL, LABEL_COL, 'proxy_label', 'Account.1']
                    and combined[c].dtype.name in ('category', 'object')]

    for c in cat_features:
        if combined[c].dtype.name != 'category':
            combined[c] = combined[c].astype('category')

    # 2. cont_features = everything else
    cont_features = [c for c in combined.columns 
                    if c not in [IBAN_COL, TS_COL, LABEL_COL, 'proxy_label', 'is_first_event', 'Account.1'] + cat_features]

    EVENT_FEATURES_ORDERED = cont_features + cat_features

    combined = combined.sort_values([IBAN_COL, TS_COL]).reset_index(drop=True)
    combined['is_first_event'] = combined.groupby(IBAN_COL).cumcount() == 0
    combined[cont_features] = combined[cont_features].fillna(0)

    for c in cat_features:
        if '__MISSING__' not in combined[c].cat.categories:
            combined[c] = combined[c].cat.add_categories('__MISSING__')
        combined[c] = combined[c].fillna('__MISSING__')

    # --- vectorized lag1 (replaces the per-group Python loop) ---
    lag = combined.groupby(IBAN_COL)[EVENT_FEATURES_ORDERED].shift(1)
    lag.columns = [f'{f}_lag1' for f in EVENT_FEATURES_ORDERED]

    out = pd.concat(
        [lag, combined[[LABEL_COL, IBAN_COL, TS_COL, 'is_first_event']]],
        axis=1
    )
    out = out.loc[~out['is_first_event']].reset_index(drop=True)
    # --- end vectorized block ---

    cat_lag_cols = [f'{c}_lag1' for c in cat_features]
    num_lag_cols = [c for c in out.columns
                    if c not in cat_lag_cols + [LABEL_COL, IBAN_COL, TS_COL, 'is_first_event']]

    for c in cat_lag_cols:
        out[c] = out[c].fillna('__MISSING__').astype(str).astype('category')
    for c in num_lag_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    return out.drop(columns='is_first_event'), num_lag_cols, cat_lag_cols, cat_features

In [ ]:
combined = combined.sort_values(TS_COL).reset_index(drop=True)
combined[TS_COL] = pd.to_datetime(combined[TS_COL], format='mixed', errors='coerce')
n_bad = combined[TS_COL].isna().sum()
if n_bad:
    print(f"Dropping {n_bad} rows with unparseable timestamps")
combined = combined.dropna(subset=[TS_COL])
combined[LABEL_COL] = combined[LABEL_COL].map({'Fraudeur': 1, 'Faux Positif': 0}).astype(float)
print(combined[LABEL_COL].unique())
print(combined[LABEL_COL].dtype)
n_splits = 5
t_min, t_max = combined[TS_COL].min(), combined[TS_COL].max()
edges = pd.date_range(t_min, t_max, periods=n_splits + 2)  # +2 -> n_splits test windows + initial train seed

gap = pd.Timedelta(hours=1)  # tune to your delta.t scale

train_dfs = []
test_dfs = []

for fold in range(n_splits):
    train_end = edges[fold + 1]
    test_start = train_end + gap
    test_end = edges[fold + 2]

    train_df = combined[combined[TS_COL] < train_end].assign(fold=fold)
    test_df  = combined[(combined[TS_COL] >= test_start) & (combined[TS_COL] < test_end)].assign(fold=fold)

    train_dfs.append(train_df)
    test_dfs.append(test_df)

    print(f"Fold {fold}")
    print(f"  train: {train_df[TS_COL].min()} – {train_df[TS_COL].max()} ({len(train_df)} rows)")
    print(f"  test : {test_df[TS_COL].min()} – {test_df[TS_COL].max()} ({len(test_df)} rows)")
    print(f"  train fraud rate: {train_df[LABEL_COL].mean():.4f}")
    print(f"  test  fraud rate: {test_df[LABEL_COL].mean():.4f}")
    print()

train_df = pd.concat(train_dfs, ignore_index=True)
test_df  = pd.concat(test_dfs, ignore_index=True)
    # fit/predict/eval here per fold

In [ ]:
# Avg NA per Row 
combined.isna().mean(axis=1).mean()

In [ ]:
test_df.head()

In [ ]:
#Check if the val set is representative at least class wise
print(test_df[LABEL_COL].value_counts(normalize=True))
print(train_df[LABEL_COL].value_counts(normalize=True))
print(len(train_df))
print(len(test_df))

In [ ]:
def cumulative_nunique_vectorized(df, group_col, value_col):
    dup = df.duplicated(subset=[group_col, value_col])
    first_seen = (~dup).astype(int)
    return first_seen.groupby(df[group_col]).cumsum()

def get_item(bic, d, str_key):
    entry = d.get(str(bic) if not pd.isna(bic) else 'nan')
    return entry[str_key] if entry else float('nan')

def features_creator(dataset):
    # sort once: per-account chronological order (matches original per-group sort)
    dataset = dataset.sort_values(['Account', 'Timestamp']).reset_index(drop=True)
    dataset['Timestamp'] = pd.to_datetime(dataset['Timestamp'])

    # --- Cross-bank reliability features ---
    for bank_col, prefix in [('From Bank', 'declaring'), ('To Bank', 'holding')]:
        bank_key = dataset[bank_col].fillna('UNKNW')
        grp = dataset.groupby(bank_key)

        is_fraud = (dataset[LABEL_COL] == 'Fraudeur').astype(int)
        is_fp    = (dataset[LABEL_COL] == 'Faux Positif').astype(int)

        prior_fraud_cnt = is_fraud.groupby(bank_key).cumsum().groupby(bank_key).shift(1)
        prior_fp_cnt    = is_fp.groupby(bank_key).cumsum().groupby(bank_key).shift(1)
        prior_n         = grp.cumcount()

        dataset[f'{prefix}.fraud_rate']    = (prior_fraud_cnt / prior_n.replace(0, np.nan)).fillna(0)
        dataset[f'{prefix}.fp_rate']       = (prior_fp_cnt / prior_n.replace(0, np.nan)).fillna(0)
        dataset[f'{prefix}.nb.prior.txn']  = prior_n
        dataset[f'{prefix}.has.history']   = (prior_n > 0).astype(int)

    # --- Corridor causal fraud rate ---
    corridor_key = dataset['From Bank'].fillna('UNKNW').astype(str) + '_' + dataset['To Bank'].fillna('UNKNW').astype(str)
    is_fraud = (dataset[LABEL_COL] == 'Fraudeur').astype(int)
    corridor_prior_fraud = is_fraud.groupby(corridor_key).cumsum().groupby(corridor_key).shift(1)
    corridor_prior_n     = dataset.groupby(corridor_key).cumcount()
    dataset['corridor.fraud_rate']   = (corridor_prior_fraud / corridor_prior_n.replace(0, np.nan)).fillna(0)
    dataset['corridor.nb.prior.txn'] = corridor_prior_n

    # --- Per-account features --- # less for looping and dicts
    acc_grp = dataset.groupby('Account')

    dataset['nb.currency'] = acc_grp.cumcount() + 1
    dataset['delta.t'] = acc_grp['Timestamp'].diff().dt.total_seconds().fillna(0)

    dataset['currency.mismatch'] = (dataset['Receiving Currency'] != dataset['Payment Currency']).astype(int)
    dataset['is.self.transfer']  = (dataset['Account'] == dataset['Account.1']).astype(int)
    dataset['is.intra.bank']     = (dataset['From Bank'] == dataset['To Bank']).astype(int)

    dataset['log.amount'] = np.log1p(dataset['Amount Paid'].astype(float))

    dataset['hour.of.day']  = dataset['Timestamp'].dt.hour
    dataset['day.of.week']  = dataset['Timestamp'].dt.dayofweek
    dataset['is.off.hours'] = dataset['Timestamp'].dt.hour.between(0, 5).astype(int)

    dataset['nb.distinct.to.bank_cum']   = cumulative_nunique_vectorized(dataset, 'Account', 'To Bank')
    dataset['nb.distinct.from.bank_cum'] = cumulative_nunique_vectorized(dataset, 'Account', 'From Bank')
    dataset['nb.distinct.payfmt_cum']    = cumulative_nunique_vectorized(dataset, 'Account', 'Payment Format')

    # --- Holding PSP (keyed by To Bank) ---
    dict_bic_holding = {}
    for x, obj in dataset.groupby('To Bank'):
        vc_2 = obj['Receiving Currency'].value_counts()
        vc_3 = obj['Payment Currency'].value_counts()
        dict_bic_holding[str(x)] = {
            'top_RC': vc_2.index[0],
            'top_SC': vc_3.index[0],
            'nb.events.holding': obj['Timestamp'].nunique(),
            'nb.iban.holding': obj['Account'].nunique()
        }

    # --- Declaring PSP (keyed by From Bank) ---
    dict_bic_declaring = {}
    for x, obj in dataset.fillna({'From Bank': 'UNKNW'}).groupby('From Bank'):
        vc_2 = obj['Receiving Currency'].value_counts()
        vc_3 = obj['Payment Currency'].value_counts()
        dict_bic_declaring[str(x)] = {
            'top_RC': vc_2.index[0],
            'top_SC': vc_3.index[0],
            'nb.events.declaring': obj['Timestamp'].nunique(),
            'nb.iban.declaring': obj['Account'].nunique()
        }

    # --- Row-wise lookups via map --- #thx Claude
    to_bank_str = dataset['To Bank'].apply(lambda b: str(b) if not pd.isna(b) else 'nan')
    from_bank_str = dataset['From Bank'].apply(lambda b: str(b) if not pd.isna(b) else 'nan')

    dataset['top.1.holder.RC']   = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'top_RC'))
    dataset['top.1.holder.SC']   = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'top_SC'))
    dataset['nb.iban.holder']    = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'nb.iban.holding'))
    dataset['nb.events.holder']  = to_bank_str.map(lambda k: get_item(k, dict_bic_holding, 'nb.events.holding'))

    dataset['top.1.declaring.RC']    = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'top_RC'))
    dataset['top.1.declaring.SC']    = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'top_SC'))
    dataset['nb.iban.declaring']     = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'nb.iban.declaring'))
    dataset['nb.events.declaring']   = from_bank_str.map(lambda k: get_item(k, dict_bic_declaring, 'nb.events.declaring'))

    del dict_bic_declaring, dict_bic_holding

    # --- Fan in/out ---
    fan_out = dataset.groupby('Account')['Account.1'].nunique().rename('fan.out')
    fan_in  = dataset.groupby('Account.1')['Account'].nunique().rename('fan.in')

    dataset['fan.out'] = dataset['Account'].map(fan_out).fillna(0)
    dataset['fan.in']  = dataset['Account'].map(fan_in).fillna(0)
    dataset['fan.ratio'] = dataset['fan.in'] / (dataset['fan.out'] + 1)

    dataset['key'] = dataset['Account']

    return dataset

In [ ]:
train_df = features_creator(train_df)
test_df = features_creator(test_df)
train_df, num_lag_cols, cat_lag_cols, cat_features = create_sliding(train_df)
test_df, num_lag_cols, cat_lag_cols, cat_features = create_sliding(test_df)

In [ ]:
test_df.head()

In [ ]:

drop_cols = [LABEL_COL, IBAN_COL, TS_COL,'key_lag1','Unnamed: 0_lag1','fold_lag1','Amount Received_lag1', 'Amount Paid_lag1']
num_lag_cols = [c for c in num_lag_cols if c != 'key_lag1'] #Not a training feature
cat_lag_cols = [c for c in cat_lag_cols if c != 'Amount Received_lag1'] #Not normalized
cat_lag_cols = [c for c in cat_lag_cols if c != 'Amount Paid_lag1'] #Not normalized
cat_lag_cols = [c for c in cat_lag_cols if c != 'key_lag1'] #Not a training feature
num_lag_cols = [c for c in num_lag_cols if c != 'Unnamed: 0_lag1'] #Not a training feature
num_lag_cols = [c for c in num_lag_cols if c != 'fold_lag1'] #Not a training feature and leak


print(cat_lag_cols)
print(num_lag_cols)
X_raw   = train_df.drop(columns=drop_cols)

X_raw.head()


In [ ]:
print(X_raw.dtypes)

In [ ]:

y_train = train_df[LABEL_COL]

# --- categorical: codes, with +1 shift so 0 = "unknown/missing" ---
for col in cat_lag_cols:
    X_raw[col] = X_raw[col].astype('category')

train_categories = {col: X_raw[col].cat.categories for col in cat_lag_cols}

for col in cat_lag_cols:
    X_raw[col] = X_raw[col].cat.codes + 1   # shift: -1 (missing) -> 0

cat_cardinalities = [int(X_raw[col].max()) + 1 for col in cat_lag_cols] # +1 already includes the unknown bucket

all_nan_cols = X_raw[num_lag_cols].columns[X_raw[num_lag_cols].isna().all()]
partial_cols = [c for c in num_lag_cols if c not in all_nan_cols]

imputer = SimpleImputer(strategy='median')
X_cont_partial = imputer.fit_transform(X_raw[partial_cols])

X_cont = pd.DataFrame(X_cont_partial, columns=partial_cols, index=X_raw.index)
for col in all_nan_cols:
    X_cont[col] = 0.0  #Fallback

X_cont = X_cont[num_lag_cols]  

scaler  = StandardScaler().fit(X_cont)
X_cont  = scaler.transform(X_cont).astype('float32')

X_cat = X_raw[cat_lag_cols].astype('float32').values
X_train_np = np.hstack([X_cont, X_cat]).astype('float32')


X_raw_test = test_df.drop(columns=drop_cols)
y_test     = test_df[LABEL_COL]

for col in cat_lag_cols:
    X_raw_test[col] = pd.Categorical(
        X_raw_test[col], categories=train_categories[col]
    ).codes + 1 #fallback + col

X_cont_test_partial = imputer.transform(X_raw_test[partial_cols])
X_cont_test = pd.DataFrame(X_cont_test_partial, columns=partial_cols, index=X_raw_test.index)
for col in all_nan_cols:
    X_cont_test[col] = 0.0
X_cont_test = X_cont_test[num_lag_cols]
X_cont_test = scaler.transform(X_cont_test).astype('float32')

X_cat_test  = X_raw_test[cat_lag_cols].astype('float32').values

X_test_np = np.hstack([X_cont_test, X_cat_test]).astype('float32')

In [ ]:
print(f"Classes in train {np.unique(y_train)}")
print(f"Train first 3 rows : {X_train_np[:3,]}") # Sanity check 
print(f"shape of the np obj: {np.shape(X_train_np)}")
print(f"Number of cols from the pd df: {len(train_df.drop(columns=drop_cols).columns)}")
used_cols = set(num_lag_cols) | set(cat_lag_cols)
missing = set(X_raw.columns) - used_cols
print(missing)

In [ ]:
train_df.to_parquet('train_df.parquet')
test_df.to_parquet('test_df.parquet')

# numpy arrays + anything else important
joblib.dump({
    'X_train_np': X_train_np,
    'y_train': y_train,
    'X_test_np': X_test_np,
    'y_test': y_test,
    'imputer': imputer,
    'scaler': scaler,
    'train_categories': train_categories,
    'all_nan_cols': all_nan_cols,
    'partial_cols': partial_cols,
    'num_lag_cols' : num_lag_cols,
    'cat_lag_cols': cat_lag_cols,
    'cat_cardinalities': cat_cardinalities,
    'cat_features': cat_features,
}, 'preprocessed.joblib')

In [ ]:
#Small model for testing
scoring = make_scorer(average_precision_score,
                      response_method="predict_proba",
                      average='macro')

deadline = DeadlineStopper(total_time=1*60)  

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train 
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = xgb.XGBClassifier(
    enable_categorical=True,
    tree_method='hist',
    eval_metric='aucpr', 
    random_state=42,
    base_score=0.5
)

search_spaces = {
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),
    'max_depth': Integer(2, 10), #max depth
    'min_child_weight': Integer(1, 5),  # lowest better for minority classes 
    'subsample': Real(0.5, 1.0),
    'colsample_bytree': Real(0.3, 1.0), # Fraction of total training data used to build each tree
    'reg_lambda': Real(1e-9, 100., prior='log-uniform'),
    'reg_alpha': Real(1e-9, 100., prior='log-uniform'),
    'n_estimators': Integer(50, 1000), #Number of trees smaller here for speed and wider search 
    'max_delta_step': Integer(3,7), # maximun weight assigned to each tree leaf good for imbalanced classes and regression 
}

opt = BayesSearchCV(
    estimator=model,
    search_spaces=search_spaces,
    scoring=scoring,
    n_iter=1000,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
) # Run a BO search for the hyperparameters to find the ebst configuration for the dataset 

opt.fit(X_train_np, y_train, **{'sample_weight': sample_weights}, callback=deadline)

print("Best score:", opt.best_score_)
print("Best params:", opt.best_params_)


In [ ]:

model = opt.best_estimator_
preds = model.predict(X_test_np)
proba = model.predict_proba(X_test_np)
print(classification_report(y_test, preds))
auc = roc_auc_score(y_test, proba[:, 1])
print(f"ROC-AUC: {auc:.4f}")

joblib.dump(model, 'xgb_features_model.pkl')

In [ ]:
proba = model.predict_proba(X_test_np)[:, 1]

# Compute ROC points and AUC
fpr, tpr, thresholds = roc_curve(y_test, proba)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'XGBoost (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# Axis formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
X_test_shap = pd.DataFrame(X_test_np, columns =  test_df.drop(columns=drop_cols).columns)
X_train_shap = pd.DataFrame(X_train_np, columns =  test_df.drop(columns=drop_cols).columns)
for c in X_test_shap.select_dtypes('category').columns:
    X_test_shap[c] = X_test_shap[c].cat.codes.astype(float)

explainer = shap.TreeExplainer(model)  
shap_values_xgb = explainer(X_test_shap)
shap.summary_plot(shap_values_xgb, X_test_shap, max_display=20, show=False)
plt.show()

In [ ]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_np, y_train, sample_weight=sample_weights)

In [ ]:

model = model
preds = model.predict(X_test_np)
proba = model.predict_proba(X_test_np)
print(classification_report(y_test, preds))
auc = roc_auc_score(y_test, proba[:, 1])
print(f"ROC-AUC: {auc:.4f}")

joblib.dump(model, 'logistic_reg_features.pkl')

In [ ]:
proba = model.predict_proba(X_test_np)[:, 1]

# Compute ROC points and AUC
fpr, tpr, thresholds = roc_curve(y_test, proba)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')

# Axis formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
X_test_shap = pd.DataFrame(X_test_np, columns =  test_df.drop(columns=drop_cols).columns)
X_train_shap = pd.DataFrame(X_train_np, columns =  test_df.drop(columns=drop_cols).columns)
for c in X_test_shap.select_dtypes('category').columns:
    X_test_shap[c] = X_test_shap[c].cat.codes.astype(float)

explainer = shap.Explainer(model, X_test_shap)   # auto-selects TreeExplainer
shap_values_rf = explainer(X_test_shap)
shap.summary_plot(shap_values_rf, X_test_shap, max_display=20, show=False)
plt.show()